In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Mejos"
STUDENT_ID = "TUPM-26-3605"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

raw_measurements = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)

# Missing, constant, and usable features (same method as Task 9)
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
retained_features = [feature_names[i] for i in usable_indices]
raw_usable = raw_measurements[:, usable_indices].copy()

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)
print("Usable features:", len(retained_features))

Student: Mejos TUPM-26-3605
Seed Digit: 5 | ID Sum: 22 | Random Seed: 263605
Usable features: 474


In [2]:
# 1. Five usable features with at least 10 observed values in each outcome group
pass_rows = np.flatnonzero(labels == -1)
fail_rows = np.flatnonzero(labels == 1)

pass_obs = (~np.isnan(raw_usable[pass_rows])).sum(axis=0)
fail_obs = (~np.isnan(raw_usable[fail_rows])).sum(axis=0)
eligible = np.flatnonzero((pass_obs >= 10) & (fail_obs >= 10))

rng = np.random.default_rng(RANDOM_SEED)
chosen = rng.choice(eligible, size=5, replace=False)
selected = [retained_features[i] for i in chosen]

# 2. Two preparation strategies
X_avail = raw_usable[:, chosen]                                    # A: available-case (keeps NaN)
medians_full = np.nanmedian(X_avail, axis=0)
X_imp = np.where(np.isnan(X_avail), medians_full, X_avail)         # B: median imputation (full dataset)

# 3. Fail minus Pass mean difference
diff_avail = np.nanmean(X_avail[fail_rows], axis=0) - np.nanmean(X_avail[pass_rows], axis=0)
diff_imp = np.nanmean(X_imp[fail_rows], axis=0) - np.nanmean(X_imp[pass_rows], axis=0)

# 4. Stratified bootstrap (resample with replacement inside each outcome group)
reps = 200 + 10 * SEED_DIGIT

def run_bootstrap(seed):
    r = np.random.default_rng(seed)
    boot_a = np.zeros((reps, 5))
    boot_i = np.zeros((reps, 5))
    for k in range(reps):
        p = r.choice(pass_rows, size=len(pass_rows), replace=True)
        f = r.choice(fail_rows, size=len(fail_rows), replace=True)
        boot_a[k] = np.nanmean(X_avail[f], axis=0) - np.nanmean(X_avail[p], axis=0)
        boot_i[k] = np.nanmean(X_imp[f], axis=0) - np.nanmean(X_imp[p], axis=0)
    return boot_a, boot_i

boot_a, boot_i = run_bootstrap(RANDOM_SEED)

# 5. Bootstrap mean difference and percentile 95% confidence interval
boot_mean_a = boot_a.mean(axis=0)
boot_mean_i = boot_i.mean(axis=0)
ci_a = np.percentile(boot_a, [2.5, 97.5], axis=0)
ci_i = np.percentile(boot_i, [2.5, 97.5], axis=0)

# 6. Compare direction, confidence intervals, and rankings
robust = pd.DataFrame({
    "Feature": selected,
    "Diff_Avail": diff_avail,
    "Boot_Mean_Avail": boot_mean_a,
    "CI_Low_Avail": ci_a[0],
    "CI_High_Avail": ci_a[1],
    "Diff_Imp": diff_imp,
    "Boot_Mean_Imp": boot_mean_i,
    "CI_Low_Imp": ci_i[0],
    "CI_High_Imp": ci_i[1],
    "Abs_Change": np.abs(diff_imp - diff_avail),
    "Sign_Changed": np.sign(diff_avail) != np.sign(diff_imp),
    "Abs_Diff_Avail": np.abs(diff_avail),
    "Abs_Diff_Imp": np.abs(diff_imp)
})

order_a = robust.sort_values(by="Abs_Diff_Avail", ascending=False)["Feature"].tolist()
order_i = robust.sort_values(by="Abs_Diff_Imp", ascending=False)["Feature"].tolist()
robust["Rank_Avail"] = [order_a.index(f) + 1 for f in selected]
robust["Rank_Imp"] = [order_i.index(f) + 1 for f in selected]
robust["Rank_Changed"] = robust["Rank_Avail"] != robust["Rank_Imp"]
robust["CI_Excludes_Zero_Avail"] = (robust["CI_Low_Avail"] > 0) | (robust["CI_High_Avail"] < 0)
robust["CI_Excludes_Zero_Imp"] = (robust["CI_Low_Imp"] > 0) | (robust["CI_High_Imp"] < 0)

largest = robust.sort_values(by="Abs_Change", ascending=False).iloc[0]
sign_changed = robust.loc[robust["Sign_Changed"], "Feature"].tolist()
rank_changed = robust.loc[robust["Rank_Changed"], "Feature"].tolist()

# 7. Reproducibility check with the same seed, then export
boot_a2, boot_i2 = run_bootstrap(RANDOM_SEED)
reproducible = np.array_equal(boot_a, boot_a2) and np.array_equal(boot_i, boot_i2)

robust.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

report = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT\n"
    f"Student: {LAST_NAME} {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Selected Features: {', '.join(selected)}\n"
    f"Bootstrap Repetitions: {reps}\n"
    f"Largest Change in Mean Difference: {largest['Feature']} ({largest['Abs_Change']:.4f})\n"
    f"Features with Changed Sign: {', '.join(sign_changed) if sign_changed else 'None'}\n"
    f"Features with Changed Ranking: {', '.join(rank_changed) if rank_changed else 'None'}\n"
    f"Reproducibility Verified: {reproducible}\n"
    "Interpretation Notes:\n"
    "- Mean differences describe groups; they do not show that a feature causes failure.\n"
    "- A confidence interval that includes zero means the difference is not clearly separated from zero.\n"
    "- Features whose sign or rank changes between strategies are sensitive to missing-data handling.\n"
)
(OUTPUT_DIR / "sensitivity_report.txt").write_text(report, encoding="utf-8")

print("Selected Features:", selected)
print("Bootstrap Repetitions:", reps)
print(robust.round(4))
print("Largest Change in Mean Difference:", largest["Feature"], f"{largest['Abs_Change']:.4f}")
print("Features with Changed Difference Sign:", sign_changed)
print("Features with Changed Ranking:", rank_changed)
print("Reproducibility Verified:", reproducible)
for name in ["robustness_comparison.csv", "sensitivity_report.txt"]:
    print(name, "Created:", (OUTPUT_DIR / name).is_file())

Selected Features: ['Feature_062', 'Feature_285', 'Feature_545', 'Feature_025', 'Feature_338']
Bootstrap Repetitions: 250
       Feature  Diff_Avail  Boot_Mean_Avail  CI_Low_Avail  CI_High_Avail  \
0  Feature_062     -0.6578          -0.7389       -2.8229         1.2911   
1  Feature_285     -0.0477          -0.0483       -0.2195         0.1219   
2  Feature_545     -0.2426          -0.2408       -0.4128        -0.0447   
3  Feature_025     -0.0262          -0.0249       -0.0685         0.0218   
4  Feature_338      0.1545           0.1660       -0.4886         1.2363   

   Diff_Imp  Boot_Mean_Imp  CI_Low_Imp  CI_High_Imp  Abs_Change  Sign_Changed  \
0   -0.6564        -0.7376     -2.8219       1.2930      0.0014         False   
1   -0.0472        -0.0478     -0.2188       0.1227      0.0005         False   
2   -0.2419        -0.2401     -0.4118      -0.0443      0.0007         False   
3   -0.0252        -0.0239     -0.0686       0.0219      0.0009         False   
4    0.1567     